In [1]:
import getpass
import json
import sys
from typing import Annotated, Sequence, TypedDict, Literal
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Securely prompt for the key string as an in-memory runtime variable
anthropic_api_key = getpass.getpass("Enter your Anthropic API Key: ")

print("✅ Environment initialized. Credentials securely isolated in volatile memory.")

✅ Environment initialized. Credentials securely isolated in volatile memory.


In [2]:
# 1. DEFINE THE CENTRAL GLOBAL STATE SCHEMA
class OrchestratorState(TypedDict):
    user_request: str           # Original raw objective
    current_plan: str           # Strategy roadmap output from the Planner
    research_data: str          # Facts collected by the Researcher
    draft_content: str          # Document written by the Writer
    review_feedback: str        # Quality evaluation logs from the Reviewer
    reviewer_status: str        # Flags: 'REJECTED' or 'READY FOR HUMAN REVIEW'
    human_feedback: str         # Inputs received through the notebook input gate
    next_node: str              # Internal tracking variable for the Supervisor router

# 2. DEFINE SYSTEM ENGINE TOOLS
@tool
def web_search(query: str) -> str:
    """Queries live internet indexes to capture verifiable facts, metrics, and reference details."""
    print(f"\n🔍 [RESEARCHER TOOL -> WEB SEARCH]: Looking up data for: '{query}'")
    
    if "agentic ai" in query.lower() or "langgraph" in query.lower():
        return json.dumps({
            "status": "success",
            "source": "https://tech-insights.io/agentic-trends-2026",
            "data": "Market shifts show enterprise adoption of Agentic AI architectures growing rapidly. LangGraph serves as a state-machine orchestrator, using native checkpointers for flawless Human-in-the-Loop interventions and transaction fault-tolerance."
        })
    return "Search completed. Information aggregated into state vector."

tool_registry = {"web_search": web_search}

In [3]:
# Initialize Claude 3.5 Sonnet core engine
llm_agent = ChatAnthropic(
    model="claude-sonnet-4-6", 
    temperature=0.2, 
    anthropic_api_key=anthropic_api_key
)

def planner_node(state: OrchestratorState) -> dict:
    print("\n📋 [NODE -> PLANNER]: Developing logical research agenda roadmap...")
    prompt = f"""You are an elite strategic Project Manager. Break down the user's core request into a structured roadmap.
    User Request: {state['user_request']}
    Any Human Feedback to fix: {state.get('human_feedback', 'None')}
    
    Output a clear, bulleted outline detailing exactly what topics to research and how to format the final paper.
    Format cleanly with Markdown headers."""
    
    response = llm_agent.invoke([HumanMessage(content=prompt)])
    return {"current_plan": response.content}

def researcher_node(state: OrchestratorState) -> dict:
    print("\n🔬 [NODE -> RESEARCHER]: Consulting data registries and verifying parameters...")
    search_query = "Agentic AI LangGraph enterprise adoption 2026"
    tool_observation = web_search.invoke({"query": search_query})
    
    prompt = f"""You are a Data Retrieval Specialist. Analyze this raw tool data payload: {tool_observation}
    Cross-reference it with the project layout plan: {state['current_plan']}
    Extract and summarize the precise statistics and source URLs into a clean, factual data briefing."""
    
    response = llm_agent.invoke([HumanMessage(content=prompt)])
    return {"research_data": response.content}

def writer_node(state: OrchestratorState) -> dict:
    print("\n✍️ [NODE -> WRITER]: Synthesizing facts into business narrative...")
    prompt = f"""You are an Expert Technical Content Creator. Write an executive-grade business report using Markdown headers.
    Follow the blueprint: {state['current_plan']}
ground your claims ONLY in these researched facts: {state['research_data']}"""
    
    response = llm_agent.invoke([HumanMessage(content=prompt)])
    return {"draft_content": response.content}

def reviewer_node(state: OrchestratorState) -> dict:
    print("\n⚖️ [NODE -> REVIEWER]: Auditing draft integrity against constraints...")
    prompt = f"""You are an Editorial Quality Assurance Judge. Evaluate the draft content against the collected research data.
    Draft Content: {state['draft_content']}
    Verified Facts: {state['research_data']}
    
    Check for flow, clarity, and accuracy. At the absolute end of your output, you MUST append exactly one of these status tokens:
    If it is excellent: [STATUS: READY FOR HUMAN REVIEW]
    If it has errors: [STATUS: REJECTED]"""
    
    response = llm_agent.invoke([HumanMessage(content=prompt)])
    output_text = response.content
    status_flag = "READY FOR HUMAN REVIEW" if "[STATUS: READY FOR HUMAN REVIEW]" in output_text else "REJECTED"
    
    return {"review_feedback": output_text, "reviewer_status": status_flag}

In [4]:
# Add this to the bottom of Cell 3
def human_gate_node(state: OrchestratorState) -> dict:
    """A blank placeholder node that acts as a secure landing spot for the human interrupt."""
    print("\n🛑 [NODE -> HUMAN GATE]: Graph state paused. Awaiting human input parameters...")
    return {}

In [5]:
def supervisor_router(state: OrchestratorState) -> Literal["planner", "researcher", "writer", "reviewer", "human_gate", "__end__"]:
    print("\n🔀 [SUPERVISOR]: Inspecting state parameters to calculate routing path...")
    
    # 1. Sequential Pipeline Processing
    if not state.get("current_plan"):
        return "planner"
    if not state.get("research_data"):
        return "researcher"
    if not state.get("draft_content"):
        return "writer"
    if not state.get("review_feedback"):
        return "reviewer"
    
    # 2. Evaluation / Human Intervention Branching
    if state.get("reviewer_status") == "REJECTED":
        return "writer"
        
    # If the report is ready and the human hasn't signed off yet, route to our registered node
    if state.get("reviewer_status") == "READY FOR HUMAN REVIEW" and not state.get("human_feedback"):
        return "human_gate"
    
    # 3. Post-Human Feedback Processing Logic
    if state.get("human_feedback"):
        if "approved" in state["human_feedback"].lower():
            return "__end__"
        else:
            return "planner"
            
    return "__end__"

# Assemble the state machine blueprint
builder = StateGraph(OrchestratorState)

# Register Workers AND the new Human Gate node
builder.add_node("planner", planner_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)
builder.add_node("reviewer", reviewer_node)
builder.add_node("human_gate", human_gate_node) # FIX: node is now registered!

# Map conditional routing intersections
builder.add_conditional_edges(START, supervisor_router)
builder.add_conditional_edges("planner", supervisor_router)
builder.add_conditional_edges("researcher", supervisor_router)
builder.add_conditional_edges("writer", supervisor_router)
builder.add_conditional_edges("reviewer", supervisor_router)

# Allow the human gate to loop back into the supervisor after an update is received
builder.add_conditional_edges("human_gate", supervisor_router)

# Bind checkpointer and hook the interrupt exactly before the human gate node executes
memory_gate = MemorySaver()
compiled_orchestrator = builder.compile(checkpointer=memory_gate, interrupt_before=["human_gate"])

print("🎯 Centralized Supervisor Architecture compiled cleanly. Validation error resolved!")

🎯 Centralized Supervisor Architecture compiled cleanly. Validation error resolved!


In [6]:
session_config = {"configurable": {"thread_id": "hitl_report_run_fixed"}}
user_prompt = "Generate a comprehensive business strategy overview analyzing enterprise adoption trends of Agentic AI architectures in 2026."

initial_input = {"user_request": user_prompt}

print("🚀 Step 1: Initializing Agent Engine Operations...")
try:
    # Invoke will run smoothly and pause automatically *right before* hitting 'human_gate'
    compiled_orchestrator.invoke(initial_input, config=session_config)
except Exception as e:
    print(f"Interception notice: {e}")

# Fetch the active snapshot state out of the transaction ledger database
snapshot = compiled_orchestrator.get_state(config=session_config)

if snapshot.next and "human_gate" in snapshot.next[0]:
    print("\n" + "=" * 70)
    print("🛑 HUMAN-IN-THE-LOOP INTERRUPT TRIGGERED")
    print("=" * 70)
    print(f"REVIEWER DIAGNOSTIC LOG:\n{snapshot.values.get('review_feedback')}\n")
    print(f"DRAFT CONTENT PREVIEW:\n{snapshot.values.get('draft_content')}\n")
    print("-" * 70)
    
    # Capture input string straight from the notebook UI layout
    user_action = input("Type 'APPROVED' to finalize the report, or enter modifications to send back to the planner: ")
    
    # Update the thread dictionary state.
    if user_action.strip().upper() == "APPROVED":
        compiled_orchestrator.update_state(
            session_config,
            {"human_feedback": "APPROVED"},
            as_node="human_gate"
        )
    else:
        # If user writes changes, clear out older keys so the supervisor re-runs them down the line
        compiled_orchestrator.update_state(
            session_config,
            {
                "human_feedback": user_action, 
                "current_plan": "", 
                "research_data": "", 
                "draft_content": "", 
                "review_feedback": "",
                "reviewer_status": ""
            },
            as_node="human_gate"
        )
        
    print("\n" + "=" * 70)
    print("🚀 Step 2: Resuming Workflow Execution via Checkpoint Storage Thread...")
    print("=" * 70)
    
    # Resuming with None automatically fetches from the save state point
    final_state = compiled_orchestrator.invoke(None, config=session_config)
    
    print("\n" + "=" * 70)
    print("🏁 FINAL SYSTEM EXECUTION SUMMARY")
    print("=" * 70)
    print(f"Final human feedback recorded: {final_state.get('human_feedback')}")
    print("Process successfully hit the absolute terminal closing point (__end__).")

🚀 Step 1: Initializing Agent Engine Operations...

🔀 [SUPERVISOR]: Inspecting state parameters to calculate routing path...

📋 [NODE -> PLANNER]: Developing logical research agenda roadmap...

🔀 [SUPERVISOR]: Inspecting state parameters to calculate routing path...

🔬 [NODE -> RESEARCHER]: Consulting data registries and verifying parameters...

🔍 [RESEARCHER TOOL -> WEB SEARCH]: Looking up data for: 'Agentic AI LangGraph enterprise adoption 2026'

🔀 [SUPERVISOR]: Inspecting state parameters to calculate routing path...

✍️ [NODE -> WRITER]: Synthesizing facts into business narrative...

🔀 [SUPERVISOR]: Inspecting state parameters to calculate routing path...

⚖️ [NODE -> REVIEWER]: Auditing draft integrity against constraints...

🔀 [SUPERVISOR]: Inspecting state parameters to calculate routing path...

🛑 HUMAN-IN-THE-LOOP INTERRUPT TRIGGERED
REVIEWER DIAGNOSTIC LOG:
# Editorial Quality Assurance Evaluation

## Document: Enterprise Adoption of Agentic AI Architectures: A 2026 Business S